# Классификация рукописных цифр из набора MNIST

In [10]:
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

## Данные: MNIST

60k train / 10k test изображений 28×28, 10 классов — цифры 0–9. `ToTensor` приводит пиксели 0–255 к тензорам 0–1; `DataLoader` выдаёт батчи по 64.

In [11]:
image_path = './'

transform = transforms.Compose([
    transforms.ToTensor()
]) 

mnist_train_dataset = torchvision.datasets.MNIST(
    root=image_path, train=True,
    transform=transform, download=False
)

mnist_test_dataset = torchvision.datasets.MNIST(
    root=image_path, train=False,
    transform=transform, download=False
)

batch_size = 64
torch.manual_seed(1) 

train_dl = DataLoader(mnist_train_dataset, 
                      batch_size, shuffle=True)


In [12]:
from torch import nn

hidden_units = [32, 16]
image_size = mnist_train_dataset[0][0].shape
input_size = image_size[0] * image_size[1] * image_size[2]

all_layers = [nn.Flatten()]

for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 10))
model = nn.Sequential(*all_layers)
model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=32, bias=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Linear(in_features=16, out_features=10, bias=True)
)

## Архитектура

Flatten (28×28 = 784) → Linear(784→32) → ReLU → Linear(32→16) → ReLU → Linear(16→10). Выход без активации — logits, уходят в loss.

In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.manual_seed(1)
num_epochs = 5

for epoch in range(num_epochs):
    accuracy_list_train = 0
    
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
        accuracy_list_train += is_correct.sum()
    
    accuracy_list_train /= len(train_dl.dataset)
    print(f"Эпоха {epoch} Точность {accuracy_list_train:.4f}")

Эпоха 0 Точность 0.9556
Эпоха 1 Точность 0.9593
Эпоха 2 Точность 0.9623
Эпоха 3 Точность 0.9663
Эпоха 4 Точность 0.9677


In [15]:
pred = model(mnist_test_dataset.data / 255.)
is_correct = (
    torch.argmax(pred, dim=1) ==
    mnist_test_dataset.targets
).float()
print(f'Точность при тестировании: {is_correct.mean():.4f}')

Точность при тестировании: 0.9596
